# TrackLink USBL — XYZ vs Measured Bearing and Range

Loads a batch of parsed TrackLink fix files and, for each deployment, compares
the reported bearing and slant range against values derived from the target XYZ
offsets. Only deployments that have non-NaN `target_x/y/z` columns are
processed.

Assumed sensor-frame convention: **X = forward, Y = starboard, Z = down**.

Derived quantities:
- **Bearing**: `atan2(Y, X)` — degrees from sensor forward toward starboard
- **Slant range**: `sqrt(X² + Y² + Z²)` — metres
- **Inclination**: `atan2(Z, sqrt(X² + Y²))` — degrees below horizontal,
  positive downward

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Configuration

In [ ]:
LABELS: list[str] = [
    "qdch0ftq_20100428_020202",
    "qdch0ftq_20110415_020103",
    "qdch0ftq_20120430_002423",
    "qdch0ftq_20130406_023610",
    "qdch0ftq_20140327_071251",
]

# qd61g27j_20100421_022145
# qd61g27j_20110410_011202
# qd61g27j_20120422_043114
# qd61g27j_20130414_013620

# qdc5ghs3_20100430_024508
# qdc5ghs3_20120501_033336
# qdc5ghs3_20130405_103429

# qdch0ftq_20100428_020202
# qdch0ftq_20110415_020103
# qdch0ftq_20120430_002423
# qdch0ftq_20130406_023610
# qdch0ftq_20140327_071251

# qdchdmy1_20110416_005411
# qdchdmy1_20130406_081713
# qdchdmy1_20140328_063358

# qtqxshxs_20110815_102540
# qtqxshxs_20150327_015552
# qtqxshxs_20150328_000850
# qtqxshxs_20150328_042551

# r234xgje_20100604_230524
# r234xgje_20120530_064545
# r234xgje_20140616_205232

# r23685bc_20100605_021022
# r23685bc_20120530_233021
# r23685bc_20140616_225022

# r23m7ms0_20100606_001908
# r23m7ms0_20120601_070118
# r23m7ms0_20140616_044549

# r29mrd12_20090613_010853
# r29mrd12_20090613_104954
# r29mrd12_20110612_045149
# r29mrd12_20130611_015335

# r29mrd5h_20090612_225306
# r29mrd5h_20090613_100254
# r29mrd5h_20110612_033752
# r29mrd5h_20130611_002419

# r7jjskxq_20101023_210332
# r7jjskxq_20121013_060425
# r7jjskxq_20131022_004934

# r7jjss8n_20101023_210332
# r7jjss8n_20121013_060425
# r7jjss8n_20131022_004934

# r7jjssbh_20101023_210332
# r7jjssbh_20121013_060425
# r7jjssbh_20131022_004934

FIXES_DIR: Path = Path("/data/exos_01/acfr_tracklink_logs_v2_parsed")

## Load data

In [ ]:
REQUIRED_COLS: list[str] = [
    "timestamp",
    "ship_latitude",
    "ship_longitude",
    "ship_heading",
    "target_bearing_angle",
    "target_slant_range",
    "target_x",
    "target_y",
    "target_z",
]

fixes_by_label: dict[str, pd.DataFrame] = {}

for label in LABELS:
    path: Path = FIXES_DIR / f"{label}_tracklink_fixes.csv"
    if not path.exists():
        print(f"[skip] {label} — file not found")
        continue

    fixes: pd.DataFrame = pd.read_csv(
        path, parse_dates=["timestamp"], date_format="ISO8601"
    )

    missing: list[str] = [
        col for col in REQUIRED_COLS if col not in fixes.columns
    ]
    if missing:
        print(f"[skip] {label} — missing columns: {missing}")
        continue

    has_xyz: bool = (
        fixes[["target_x", "target_y", "target_z"]].notna().any().all()
    )
    if not has_xyz:
        print(f"[skip] {label} — target_x/y/z absent or all-NaN")
        continue

    fixes_by_label[label] = fixes
    print(f"[ok]   {label} — {len(fixes)} rows")

print(f"\nLoaded {len(fixes_by_label)} / {len(LABELS)} deployments")

## Derive bearing, slant range, and inclination from target XYZ

Sensor-frame convention: X = forward, Y = starboard, Z = down.

In [ ]:
def derive_bearing_range_inclination(
    target_x: np.ndarray,
    target_y: np.ndarray,
    target_z: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Derive bearing, slant range, and inclination from sensor-frame XYZ.

    Sensor-frame convention: X = forward, Y = starboard, Z = down.

    Arguments
    ---------
    target_x: Forward offset in metres.
    target_y: Starboard offset in metres.
    target_z: Downward offset in metres.

    Returns
    -------
    Tuple of (bearing_deg, slant_range_m, inclination_deg) where bearing is
    degrees from sensor forward toward starboard, slant_range is metres, and
    inclination is degrees below horizontal (positive downward).
    """
    horizontal_range: np.ndarray = np.sqrt(target_x**2 + target_y**2)
    slant_range_m: np.ndarray = np.sqrt(
        target_x**2 + target_y**2 + target_z**2
    )
    bearing_deg: np.ndarray = np.degrees(np.arctan2(target_y, target_x))
    inclination_deg: np.ndarray = np.degrees(
        np.arctan2(target_z, horizontal_range)
    )
    return bearing_deg, slant_range_m, inclination_deg


for label, fixes in fixes_by_label.items():
    bearing_deg, slant_range_m, inclination_deg = (
        derive_bearing_range_inclination(
            fixes["target_x"].to_numpy(),
            fixes["target_y"].to_numpy(),
            fixes["target_z"].to_numpy(),
        )
    )
    fixes["xyz_bearing_deg"] = bearing_deg
    fixes["xyz_slant_range_m"] = slant_range_m
    fixes["xyz_inclination_deg"] = inclination_deg

## Bearing axis convention variants

Four candidate axis assignments for `atan2`, to identify which sensor-frame
convention matches the measured bearing:

| Column | Formula | Assumption |
|---|---|---|
| `target_bearing_derived_1` | `atan2(Y, X)` | X = forward, Y = starboard |
| `target_bearing_derived_2` | `atan2(-Y, X)` | X = forward, Y = port |
| `target_bearing_derived_3` | `atan2(X, Y)` | X = starboard, Y = forward |
| `target_bearing_derived_4` | `atan2(-X, Y)` | X = port, Y = forward |

In [ ]:
for label, fixes in fixes_by_label.items():
    x: np.ndarray = fixes["target_x"].to_numpy()
    y: np.ndarray = fixes["target_y"].to_numpy()

    fixes["target_bearing_derived_1"] = np.degrees(np.arctan2(y, x)) % 360
    fixes["target_bearing_derived_2"] = np.degrees(np.arctan2(-y, x)) % 360
    fixes["target_bearing_derived_3"] = np.degrees(np.arctan2(x, y)) % 360
    fixes["target_bearing_derived_4"] = np.degrees(np.arctan2(-x, y)) % 360

In [ ]:
BEARING_VARIANTS: list[tuple[str, str]] = [
    ("target_bearing_derived_1", "atan2(Y, X) — X=fwd, Y=stbd"),
    ("target_bearing_derived_2", "atan2(-Y, X) — X=fwd, Y=port"),
    ("target_bearing_derived_3", "atan2(X, Y) — X=stbd, Y=fwd"),
    ("target_bearing_derived_4", "atan2(-X, Y) — X=port, Y=fwd"),
]

for label, fixes in fixes_by_label.items():
    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=[title for _, title in BEARING_VARIANTS],
        horizontal_spacing=0.12,
        vertical_spacing=0.15,
    )

    measured: pd.Series = fixes["target_bearing_angle"]

    for index, (column, title) in enumerate(BEARING_VARIANTS):
        row: int = index // 2 + 1
        col: int = index % 2 + 1

        derived: pd.Series = fixes[column]
        axis_min: float = float(min(measured.min(), derived.min()))
        axis_max: float = float(max(measured.max(), derived.max()))

        fig.add_trace(
            go.Scatter(
                x=measured,
                y=derived,
                mode="markers",
                marker=dict(size=4, color="steelblue", opacity=0.5),
                showlegend=False,
            ),
            row=row,
            col=col,
        )
        fig.add_trace(
            go.Scatter(
                x=[axis_min, axis_max],
                y=[axis_min, axis_max],
                mode="lines",
                line=dict(color="black", dash="dash", width=1),
                showlegend=False,
            ),
            row=row,
            col=col,
        )
        fig.update_xaxes(title_text="Measured (°)", row=row, col=col)
        fig.update_yaxes(title_text="Derived (°)", row=row, col=col)

    fig.update_layout(
        title=f"Bearing axis convention variants — {label}",
        height=800,
    )

    fig.show()

## Time series comparison per deployment

For each deployment: measured bearing and slant range overlaid with XYZ-derived
values, plus inclination and raw XYZ offsets.

## Scatter: measured vs XYZ-derived bearing and slant range

Each point is one fix. The dashed diagonal is the 1:1 reference line.

In [ ]:
for label, fixes in fixes_by_label.items():
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=(
            "Bearing: measured vs XYZ-derived (°)",
            "Slant range: measured vs XYZ-derived (m)",
        ),
        horizontal_spacing=0.12,
    )

    # Bearing scatter
    bearing_min: float = float(
        min(
            fixes["target_bearing_angle"].min(),
            fixes["xyz_bearing_deg"].min(),
        )
    )
    bearing_max: float = float(
        max(
            fixes["target_bearing_angle"].max(),
            fixes["xyz_bearing_deg"].max(),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=fixes["target_bearing_angle"],
            y=fixes["xyz_bearing_deg"],
            mode="markers",
            marker=dict(size=5, color="steelblue", opacity=0.6),
            name="Fixes",
            showlegend=False,
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=[bearing_min, bearing_max],
            y=[bearing_min, bearing_max],
            mode="lines",
            line=dict(color="black", dash="dash", width=1),
            name="1:1",
            showlegend=True,
        ),
        row=1,
        col=1,
    )

    # Slant range scatter
    range_min: float = float(
        min(
            fixes["target_slant_range"].min(),
            fixes["xyz_slant_range_m"].min(),
        )
    )
    range_max: float = float(
        max(
            fixes["target_slant_range"].max(),
            fixes["xyz_slant_range_m"].max(),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=fixes["target_slant_range"],
            y=fixes["xyz_slant_range_m"],
            mode="markers",
            marker=dict(size=5, color="darkorange", opacity=0.6),
            name="Fixes",
            showlegend=False,
        ),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Scatter(
            x=[range_min, range_max],
            y=[range_min, range_max],
            mode="lines",
            line=dict(color="black", dash="dash", width=1),
            name="1:1",
            showlegend=False,
        ),
        row=1,
        col=2,
    )

    fig.update_xaxes(title_text="Measured (°)", row=1, col=1)
    fig.update_yaxes(title_text="XYZ-derived (°)", row=1, col=1)
    fig.update_xaxes(title_text="Measured (m)", row=1, col=2)
    fig.update_yaxes(title_text="XYZ-derived (m)", row=1, col=2)

    fig.update_layout(
        title=f"Measured vs XYZ-derived — {label}",
        height=500,
        legend=dict(x=1.01, y=1.0),
    )

    fig.show()

In [ ]:
ROW_TITLES: list[str] = [
    "Bearing (°)",
    "Slant range (m)",
    "Inclination (°, +down)",
    "Target X — forward (m)",
    "Target Y — starboard (m)",
    "Target Z — down (m)",
]
N_ROWS: int = len(ROW_TITLES)

for label, fixes in fixes_by_label.items():
    fig = make_subplots(
        rows=N_ROWS,
        cols=1,
        shared_xaxes=True,
        subplot_titles=ROW_TITLES,
        vertical_spacing=0.05,
    )

    # Bearing
    fig.add_trace(
        go.Scatter(
            x=fixes["timestamp"],
            y=fixes["target_bearing_angle"],
            mode="lines+markers",
            marker=dict(size=3),
            name="Measured",
            line=dict(color="steelblue"),
            legendgroup="measured",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=fixes["timestamp"],
            y=fixes["xyz_bearing_deg"],
            mode="lines+markers",
            marker=dict(size=3),
            name="XYZ-derived",
            line=dict(color="tomato"),
            legendgroup="xyz",
        ),
        row=1,
        col=1,
    )

    # Slant range
    fig.add_trace(
        go.Scatter(
            x=fixes["timestamp"],
            y=fixes["target_slant_range"],
            mode="lines+markers",
            marker=dict(size=3),
            name="Measured",
            line=dict(color="steelblue"),
            legendgroup="measured",
            showlegend=False,
        ),
        row=2,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=fixes["timestamp"],
            y=fixes["xyz_slant_range_m"],
            mode="lines+markers",
            marker=dict(size=3),
            name="XYZ-derived",
            line=dict(color="tomato"),
            legendgroup="xyz",
            showlegend=False,
        ),
        row=2,
        col=1,
    )

    # Inclination
    fig.add_trace(
        go.Scatter(
            x=fixes["timestamp"],
            y=fixes["xyz_inclination_deg"],
            mode="lines+markers",
            marker=dict(size=3),
            name="Inclination",
            line=dict(color="seagreen"),
            showlegend=False,
        ),
        row=3,
        col=1,
    )

    # XYZ
    for row, (column, color) in enumerate(
        [
            ("target_x", "steelblue"),
            ("target_y", "darkorange"),
            ("target_z", "seagreen"),
        ],
        start=4,
    ):
        fig.add_trace(
            go.Scatter(
                x=fixes["timestamp"],
                y=fixes[column],
                mode="lines+markers",
                marker=dict(size=3),
                name=column,
                line=dict(color=color),
                showlegend=False,
            ),
            row=row,
            col=1,
        )

    fig.update_layout(
        title=f"XYZ vs measured bearing / range — {label}",
        height=1100,
        legend=dict(x=1.01, y=1.0),
    )

    fig.show()